# Engineer Assignment MDP Notebook 

この notebook は、アップロードされた `engineer_assignment_mdp_note.tex` の問題設定に合わせて作った **自己完結型の Colab notebook** です。  

- 状態: $s_t=(z_t,k_t)$, $z_t\in\{z_B,z_H,z_L\}$, $k_t\in\{0,\dots,K\}$
- 行動集合: $A(z_B)=\{a_H,a_L,a_T\},\ A(z_H)=A(z_L)=\{a_N\}$
- 報酬: **遷移前報酬** $r(s_t,a_t)=\mathrm{rev}(z_t)-C_w-C_{\mathrm{edu}}\mathbf{1}[a_t=a_T]$
- 配属中遷移: 継続確率 $d_H,d_L$ と、継続時の適合度低下確率 $\delta$
- 評価指標: 平均累積報酬, 平均稼働率, H/L 滞在割合, **low-fit ratio**, 平均適合度
- 評価方法: **100 ステップを 1 エピソード** とし、**複数エピソードのモンテカルロ平均**で比較
- 可視化: 方策比較に加えて、**各ステップでの $z_B/z_H/z_L$ 構成比の積み上げグラフ**を追加

注意:
- ベースライン2の $k_{safe}$ と、ベースライン3の $k_H,k_L$ は `.tex` では記号だけなので、この notebook では初期値を `k_safe=1, k_H=2, k_L=1` にしています。
- これらは後ろのセルで自由に変更できます。
- 積み上げグラフはまず見やすさを重視して、厳密な 12 状態 $(z,k)$ ではなく、粗い稼働状態 $z_B/z_H/z_L$ の構成比を描いています。

In [ ]:
# Colab では通常そのまま入っていますが、念のため最低限の依存を確認します。
import importlib
import subprocess
import sys

for pkg in ['matplotlib', 'numpy']:
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Dependencies are ready.')


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from random import Random
from statistics import mean, pstdev
from typing import Any, Dict, List, Optional, Sequence, Tuple

State = Tuple[str, int]
Action = str
Policy = List[int]


@dataclass(frozen=True)
class MDPParameters:
    """engineer_assignment_mdp_note.tex に合わせたパラメータ。"""

    K: int = 3
    gamma: float = 0.95
    R_H: float = 10.0
    R_L: float = 6.0
    C_w: float = 4.0
    C_edu: float = 1.0
    d_H: float = 0.75
    d_L: float = 0.85
    delta: float = 0.10
    p_H: Tuple[float, ...] = (0.10, 0.30, 0.55, 0.80)
    p_L: Tuple[float, ...] = (0.40, 0.60, 0.75, 0.90)
    q_T: Tuple[float, ...] = (0.70, 0.55, 0.40, 0.00)

    def validate(self) -> None:
        if self.K < 0:
            raise ValueError('K は 0 以上の整数である必要があります。')
        expected_len = self.K + 1
        for name, values in (("p_H", self.p_H), ("p_L", self.p_L), ("q_T", self.q_T)):
            if len(values) != expected_len:
                raise ValueError(f'{name} の長さは K+1={expected_len} である必要があります。')
            for value in values:
                if not 0.0 <= value <= 1.0:
                    raise ValueError(f'{name} の要素は [0,1] に入る必要があります。')
        if not 0.0 <= self.gamma < 1.0:
            raise ValueError('gamma は [0,1) の範囲で指定してください。')
        for name, value in (
            ("d_H", self.d_H),
            ("d_L", self.d_L),
            ("delta", self.delta),
        ):
            if not 0.0 <= value <= 1.0:
                raise ValueError(f'{name} は [0,1] の範囲で指定してください。')


class EngineerAssignmentMDP:
    Z_VALUES: Tuple[str, ...] = ('z_B', 'z_H', 'z_L')
    ACTIONS: Tuple[Action, ...] = ('a_H', 'a_L', 'a_T', 'a_N')

    def __init__(self, params: Optional[MDPParameters] = None):
        self.params = params or MDPParameters()
        self.params.validate()
        self.states: List[State] = [(z, k) for z in self.Z_VALUES for k in range(self.params.K + 1)]
        self.state_to_idx: Dict[State, int] = {state: idx for idx, state in enumerate(self.states)}
        self.actions: List[Action] = list(self.ACTIONS)
        self.action_to_idx: Dict[Action, int] = {action: idx for idx, action in enumerate(self.actions)}
        self.legal_actions_map: Dict[State, List[Action]] = {
            state: self._legal_actions_for_state(state) for state in self.states
        }
        self.transition_probabilities = [
            [[0.0 for _ in self.states] for _ in self.actions] for _ in self.states
        ]
        self.rewards = [
            [[0.0 for _ in self.states] for _ in self.actions] for _ in self.states
        ]
        self._build_transition_and_reward_tables()

    @property
    def num_states(self) -> int:
        return len(self.states)

    @property
    def num_actions(self) -> int:
        return len(self.actions)

    def _legal_actions_for_state(self, state: State) -> List[Action]:
        z, _ = state
        return ['a_H', 'a_L', 'a_T'] if z == 'z_B' else ['a_N']

    def get_legal_actions(self, state: State) -> List[Action]:
        return list(self.legal_actions_map[state])

    def is_action_legal(self, state: State, action: Action) -> bool:
        return action in self.legal_actions_map[state]

    def assert_legal_action(self, state: State, action: Action) -> None:
        if action not in self.action_to_idx:
            raise ValueError(f'未知の action です: {action}')
        if not self.is_action_legal(state, action):
            legal = ', '.join(self.get_legal_actions(state))
            raise ValueError(f'state={state} では action={action} は非合法です。合法 action: {legal}')

    def revenue(self, z: str) -> float:
        if z == 'z_B':
            return 0.0
        if z == 'z_H':
            return self.params.R_H
        if z == 'z_L':
            return self.params.R_L
        raise ValueError(f'未知の稼働状態です: {z}')

    def reward_from_state_action(self, state: State, action: Action) -> float:
        z, _ = state
        reward = self.revenue(z) - self.params.C_w
        if action == 'a_T':
            reward -= self.params.C_edu
        return reward

    def _set_transition(self, state: State, action: Action, next_state: State, probability: float) -> None:
        s_idx = self.state_to_idx[state]
        a_idx = self.action_to_idx[action]
        sp_idx = self.state_to_idx[next_state]
        self.transition_probabilities[s_idx][a_idx][sp_idx] += probability
        self.rewards[s_idx][a_idx][sp_idx] = self.reward_from_state_action(state, action)

    def _build_transition_and_reward_tables(self) -> None:
        p = self.params
        for state in self.states:
            z, k = state
            if z == 'z_B':
                self._set_transition(state, 'a_H', ('z_H', k), p.p_H[k])
                self._set_transition(state, 'a_H', ('z_B', k), 1.0 - p.p_H[k])

                self._set_transition(state, 'a_L', ('z_L', k), p.p_L[k])
                self._set_transition(state, 'a_L', ('z_B', k), 1.0 - p.p_L[k])

                self._set_transition(state, 'a_T', ('z_B', min(k + 1, p.K)), p.q_T[k])
                self._set_transition(state, 'a_T', ('z_B', k), 1.0 - p.q_T[k])

            elif z == 'z_H':
                self._set_transition(state, 'a_N', ('z_H', k), p.d_H * (1.0 - p.delta))
                self._set_transition(state, 'a_N', ('z_H', max(k - 1, 0)), p.d_H * p.delta)
                self._set_transition(state, 'a_N', ('z_B', k), 1.0 - p.d_H)

            elif z == 'z_L':
                self._set_transition(state, 'a_N', ('z_L', k), p.d_L * (1.0 - p.delta))
                self._set_transition(state, 'a_N', ('z_L', max(k - 1, 0)), p.d_L * p.delta)
                self._set_transition(state, 'a_N', ('z_B', k), 1.0 - p.d_L)

    def validate_transition_probabilities(self, atol: float = 1e-10) -> List[Tuple[State, Action, float]]:
        violations: List[Tuple[State, Action, float]] = []
        for state in self.states:
            s_idx = self.state_to_idx[state]
            for action in self.get_legal_actions(state):
                a_idx = self.action_to_idx[action]
                total = sum(self.transition_probabilities[s_idx][a_idx])
                if abs(total - 1.0) > atol:
                    violations.append((state, action, total))
        return violations

    def expected_return(self, state_idx: int, action_idx: int, values: Sequence[float]) -> float:
        state = self.states[state_idx]
        action = self.actions[action_idx]
        self.assert_legal_action(state, action)
        total = 0.0
        for sp_idx, prob in enumerate(self.transition_probabilities[state_idx][action_idx]):
            if prob == 0.0:
                continue
            reward = self.rewards[state_idx][action_idx][sp_idx]
            total += prob * (reward + self.params.gamma * values[sp_idx])
        return total

    def sample_next_state(self, rng: Random, state: State, action: Action) -> Tuple[State, float]:
        self.assert_legal_action(state, action)
        s_idx = self.state_to_idx[state]
        a_idx = self.action_to_idx[action]
        total_probability = sum(self.transition_probabilities[s_idx][a_idx])
        if total_probability <= 0.0:
            raise ValueError(f'state={state}, action={action} の遷移確率が設定されていません。')
        draw = rng.random()
        cumulative = 0.0
        for sp_idx, prob in enumerate(self.transition_probabilities[s_idx][a_idx]):
            cumulative += prob
            if draw <= cumulative + 1e-12:
                return self.states[sp_idx], self.rewards[s_idx][a_idx][sp_idx]
        raise RuntimeError(
            f'state={state}, action={action} のサンプリングに失敗しました。確率和={total_probability}'
        )

    def describe_problem(self) -> Dict[str, object]:
        return {
            'state': 's=(z,k), z∈{z_B,z_H,z_L}, k∈{0,...,K}',
            'actions': 'A(z_B)={a_H,a_L,a_T}, A(z_H)=A(z_L)={a_N}',
            'reward': 'r(s,a)=rev(z)-C_w-C_edu*1[a=a_T] (遷移前報酬)',
            'transition': '待機時は提案/研修の成功確率, 配属中は継続確率 d と継続時劣化 δ',
            'objective': '期待割引累積報酬の最大化',
            'evaluation_metrics': [
                '平均累積報酬(1エピソード=100 steps, 複数エピソード平均)',
                '平均稼働率',
                '高単価案件状態滞在割合',
                '低単価案件状態滞在割合',
                '低適合度滞在割合',
                '平均適合度',
                '各ステップでの z_B/z_H/z_L 構成比',
            ],
        }


def _argmax(values: List[float]) -> int:
    best_idx = 0
    best_value = values[0]
    for idx, value in enumerate(values[1:], start=1):
        if value > best_value:
            best_idx = idx
            best_value = value
    return best_idx


def _summary_stats(values: Sequence[float]) -> Dict[str, float]:
    if len(values) == 0:
        raise ValueError('summary を計算するには少なくとも 1 つの値が必要です。')
    values_list = list(values)
    mu = mean(values_list)
    sd = pstdev(values_list) if len(values_list) > 1 else 0.0
    se = sd / (len(values_list) ** 0.5) if len(values_list) > 0 else 0.0
    return {'mean': mu, 'std': sd, 'se': se}


def validate_policy(mdp: EngineerAssignmentMDP, policy: Policy) -> None:
    if len(policy) != mdp.num_states:
        raise ValueError(f'policy の長さは num_states={mdp.num_states} である必要があります。')
    for s_idx, action_idx in enumerate(policy):
        if not 0 <= action_idx < mdp.num_actions:
            raise ValueError(f'state={mdp.states[s_idx]} に対する action index が不正です: {action_idx}')
        state = mdp.states[s_idx]
        action = mdp.actions[action_idx]
        mdp.assert_legal_action(state, action)


def greedy_policy_from_values(mdp: EngineerAssignmentMDP, values: List[float]) -> Policy:
    if len(values) != mdp.num_states:
        raise ValueError(f'values の長さは num_states={mdp.num_states} である必要があります。')
    policy = [0 for _ in range(mdp.num_states)]
    for s_idx, state in enumerate(mdp.states):
        legal_indices = [mdp.action_to_idx[a] for a in mdp.get_legal_actions(state)]
        q_values = [mdp.expected_return(s_idx, a_idx, values) for a_idx in legal_indices]
        policy[s_idx] = legal_indices[_argmax(q_values)]
    return policy


def evaluate_policy(
    mdp: EngineerAssignmentMDP,
    policy: Policy,
    theta: float = 1e-10,
    max_iterations: int = 10000,
) -> List[float]:
    validate_policy(mdp, policy)
    values = [0.0 for _ in range(mdp.num_states)]
    for _ in range(max_iterations):
        delta = 0.0
        new_values = values.copy()
        for s_idx in range(mdp.num_states):
            new_values[s_idx] = mdp.expected_return(s_idx, policy[s_idx], values)
            delta = max(delta, abs(new_values[s_idx] - values[s_idx]))
        values = new_values
        if delta < theta:
            break
    return values


def value_iteration(
    mdp: EngineerAssignmentMDP,
    theta: float = 1e-10,
    max_iterations: int = 10000,
) -> Dict[str, object]:
    values = [0.0 for _ in range(mdp.num_states)]
    history: List[float] = []
    iteration = 0
    for iteration in range(max_iterations):
        delta = 0.0
        new_values = values.copy()
        for s_idx, state in enumerate(mdp.states):
            legal_indices = [mdp.action_to_idx[a] for a in mdp.get_legal_actions(state)]
            q_values = [mdp.expected_return(s_idx, a_idx, values) for a_idx in legal_indices]
            new_values[s_idx] = max(q_values)
            delta = max(delta, abs(new_values[s_idx] - values[s_idx]))
        values = new_values
        history.append(delta)
        if delta < theta:
            break
    return {
        'values': values,
        'policy': greedy_policy_from_values(mdp, values),
        'iterations': iteration + 1,
        'delta_history': history,
    }


def policy_iteration(
    mdp: EngineerAssignmentMDP,
    theta: float = 1e-10,
    max_iterations: int = 1000,
) -> Dict[str, object]:
    policy = [mdp.action_to_idx[mdp.get_legal_actions(state)[0]] for state in mdp.states]
    stable = False
    iteration = 0
    while not stable and iteration < max_iterations:
        iteration += 1
        values = evaluate_policy(mdp, policy, theta=theta)
        stable = True
        for s_idx, state in enumerate(mdp.states):
            legal_indices = [mdp.action_to_idx[a] for a in mdp.get_legal_actions(state)]
            q_values = [mdp.expected_return(s_idx, a_idx, values) for a_idx in legal_indices]
            best_action = legal_indices[_argmax(q_values)]
            if best_action != policy[s_idx]:
                stable = False
            policy[s_idx] = best_action
    values = evaluate_policy(mdp, policy, theta=theta)
    return {'values': values, 'policy': policy, 'iterations': iteration, 'stable': stable}


def policy_to_action_names(mdp: EngineerAssignmentMDP, policy: Policy) -> Dict[str, str]:
    validate_policy(mdp, policy)
    return {str(state): mdp.actions[policy[s_idx]] for s_idx, state in enumerate(mdp.states)}


def _policy_template(mdp: EngineerAssignmentMDP) -> Policy:
    return [mdp.action_to_idx[mdp.get_legal_actions(state)[0]] for state in mdp.states]


def baseline_1_myopic_utilization(mdp: EngineerAssignmentMDP) -> Policy:
    """ベースライン1: 目先稼働率最大化方策"""
    policy = _policy_template(mdp)
    for s_idx, (z, k) in enumerate(mdp.states):
        if z == 'z_B':
            policy[s_idx] = (
                mdp.action_to_idx['a_H']
                if mdp.params.p_H[k] >= mdp.params.p_L[k]
                else mdp.action_to_idx['a_L']
            )
    return policy


def baseline_2_state_safe(mdp: EngineerAssignmentMDP, k_safe: int = 1) -> Policy:
    """ベースライン2: 状態悪化回避方策"""
    policy = baseline_1_myopic_utilization(mdp)
    for s_idx, (z, k) in enumerate(mdp.states):
        if z == 'z_B' and k <= k_safe:
            policy[s_idx] = mdp.action_to_idx['a_T']
    return policy


def baseline_3_threshold(mdp: EngineerAssignmentMDP, k_H: int = 2, k_L: int = 1) -> Policy:
    """ベースライン3: 決定論的閾値方策"""
    policy = _policy_template(mdp)
    for s_idx, (z, k) in enumerate(mdp.states):
        if z == 'z_B':
            if k >= k_H:
                action = 'a_H'
            elif k_L <= k < k_H:
                action = 'a_L'
            else:
                action = 'a_T'
            policy[s_idx] = mdp.action_to_idx[action]
    return policy


def build_baseline_policies(
    mdp: EngineerAssignmentMDP,
    k_safe: int = 1,
    k_H: int = 2,
    k_L: int = 1,
) -> Dict[str, Policy]:
    return {
        'baseline_1_myopic_utilization': baseline_1_myopic_utilization(mdp),
        'baseline_2_state_safe': baseline_2_state_safe(mdp, k_safe=k_safe),
        'baseline_3_threshold': baseline_3_threshold(mdp, k_H=k_H, k_L=k_L),
    }


def simulate_policy(
    mdp: EngineerAssignmentMDP,
    policy: Policy,
    num_episodes: int = 1000,
    num_steps: int = 100,
    initial_state: Optional[State] = None,
    seed: int = 42,
) -> Dict[str, object]:
    validate_policy(mdp, policy)
    if num_episodes <= 0:
        raise ValueError('num_episodes は 1 以上である必要があります。')
    if num_steps <= 0:
        raise ValueError('num_steps は 1 以上である必要があります。')

    rng = Random(seed)
    initial_state = initial_state or ('z_B', 0)

    cumulative_rewards: List[float] = []
    utilization_rates: List[float] = []
    high_ratios: List[float] = []
    low_ratios: List[float] = []
    low_fit_ratios: List[float] = []
    avg_ks: List[float] = []

    z_state_count_by_step = {z: [0 for _ in range(num_steps)] for z in mdp.Z_VALUES}
    full_state_count_by_step = {str(state): [0 for _ in range(num_steps)] for state in mdp.states}

    low_fit_threshold = mdp.params.K / 3

    for _episode in range(num_episodes):
        state = initial_state
        cumulative_reward = 0.0
        state_counts = {'z_B': 0, 'z_H': 0, 'z_L': 0}
        low_fit_count = 0
        k_history: List[int] = []

        for t in range(num_steps):
            z, k = state
            state_counts[z] += 1
            z_state_count_by_step[z][t] += 1
            full_state_count_by_step[str(state)][t] += 1
            k_history.append(k)
            if k <= low_fit_threshold:
                low_fit_count += 1

            s_idx = mdp.state_to_idx[state]
            action = mdp.actions[policy[s_idx]]
            next_state, reward = mdp.sample_next_state(rng, state, action)

            cumulative_reward += reward
            state = next_state

        cumulative_rewards.append(cumulative_reward)
        utilization_rates.append((state_counts['z_H'] + state_counts['z_L']) / num_steps)
        high_ratios.append(state_counts['z_H'] / num_steps)
        low_ratios.append(state_counts['z_L'] / num_steps)
        low_fit_ratios.append(low_fit_count / num_steps)
        avg_ks.append(mean(k_history))

    reward_stats = _summary_stats(cumulative_rewards)
    utilization_stats = _summary_stats(utilization_rates)
    high_stats = _summary_stats(high_ratios)
    low_stats = _summary_stats(low_ratios)
    low_fit_stats = _summary_stats(low_fit_ratios)
    k_stats = _summary_stats(avg_ks)

    z_state_share_by_step = {
        z: [count / num_episodes for count in counts]
        for z, counts in z_state_count_by_step.items()
    }
    full_state_share_by_step = {
        state_label: [count / num_episodes for count in counts]
        for state_label, counts in full_state_count_by_step.items()
    }

    return {
        'num_episodes': num_episodes,
        'num_steps': num_steps,
        'initial_state': initial_state,
        'mean_cumulative_reward': reward_stats['mean'],
        'std_cumulative_reward': reward_stats['std'],
        'se_cumulative_reward': reward_stats['se'],
        'mean_utilization_rate': utilization_stats['mean'],
        'std_utilization_rate': utilization_stats['std'],
        'se_utilization_rate': utilization_stats['se'],
        'mean_high_ratio': high_stats['mean'],
        'std_high_ratio': high_stats['std'],
        'se_high_ratio': high_stats['se'],
        'mean_low_ratio': low_stats['mean'],
        'std_low_ratio': low_stats['std'],
        'se_low_ratio': low_stats['se'],
        'mean_low_fit_ratio': low_fit_stats['mean'],
        'std_low_fit_ratio': low_fit_stats['std'],
        'se_low_fit_ratio': low_fit_stats['se'],
        'mean_k': k_stats['mean'],
        'std_k': k_stats['std'],
        'se_k': k_stats['se'],
        'episode_returns': cumulative_rewards,
        'episode_utilization_rates': utilization_rates,
        'episode_high_ratios': high_ratios,
        'episode_low_ratios': low_ratios,
        'episode_low_fit_ratios': low_fit_ratios,
        'episode_mean_ks': avg_ks,
        'z_state_share_by_step': z_state_share_by_step,
        'full_state_share_by_step': full_state_share_by_step,
    }


def compare_policies(
    mdp: EngineerAssignmentMDP,
    num_episodes: int = 1000,
    num_steps: int = 100,
    initial_state: Optional[State] = None,
    seed: int = 42,
    k_safe: int = 1,
    k_H: int = 2,
    k_L: int = 1,
) -> Dict[str, Dict[str, object]]:
    vi_result = value_iteration(mdp)
    pi_result = policy_iteration(mdp)
    policies = {
        'value_iteration_optimal': vi_result['policy'],
        'policy_iteration_optimal': pi_result['policy'],
        **build_baseline_policies(mdp, k_safe=k_safe, k_H=k_H, k_L=k_L),
    }
    return {
        name: simulate_policy(
            mdp,
            policy,
            num_episodes=num_episodes,
            num_steps=num_steps,
            initial_state=initial_state,
            seed=seed,
        )
        for name, policy in policies.items()
    }


def results_to_table(results: Dict[str, Dict[str, object]]) -> List[Dict[str, object]]:
    rows = []
    for name, metrics in results.items():
        rows.append({
            'policy': name,
            'mean_cumulative_reward': metrics['mean_cumulative_reward'],
            'std_cumulative_reward': metrics['std_cumulative_reward'],
            'se_cumulative_reward': metrics['se_cumulative_reward'],
            'mean_utilization_rate': metrics['mean_utilization_rate'],
            'mean_high_ratio': metrics['mean_high_ratio'],
            'mean_low_ratio': metrics['mean_low_ratio'],
            'mean_low_fit_ratio': metrics['mean_low_fit_ratio'],
            'mean_k': metrics['mean_k'],
        })
    return sorted(rows, key=lambda row: row['mean_cumulative_reward'], reverse=True)


def plot_policy_comparison(results: Dict[str, Dict[str, object]], save_path: Optional[str] = None) -> Tuple[Any, Any]:
    import matplotlib.pyplot as plt

    metrics = [
        ('mean_cumulative_reward', 'se_cumulative_reward', 'Mean cumulative reward (100-step episode)'),
        ('mean_utilization_rate', 'se_utilization_rate', 'Mean utilization rate'),
        ('mean_high_ratio', 'se_high_ratio', 'High-state ratio'),
        ('mean_low_ratio', 'se_low_ratio', 'Low-state ratio'),
        ('mean_low_fit_ratio', 'se_low_fit_ratio', 'Low-fit ratio'),
        ('mean_k', 'se_k', 'Mean fit level k'),
    ]
    policy_names = list(results.keys())
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    flat_axes = axes.flatten()

    for ax, (metric_key, error_key, title) in zip(flat_axes, metrics):
        values = [results[name][metric_key] for name in policy_names]
        errors = [results[name][error_key] for name in policy_names]
        ax.bar(policy_names, values, yerr=errors, capsize=4)
        ax.set_title(title)
        ax.tick_params(axis='x', rotation=45)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    return fig, axes


def plot_state_composition_by_step(
    results: Dict[str, Dict[str, object]],
    policy_names: Optional[Sequence[str]] = None,
    save_path: Optional[str] = None,
) -> Tuple[Any, Any]:
    import matplotlib.pyplot as plt

    if policy_names is None:
        policy_names = list(results.keys())
    policy_names = list(policy_names)
    num_policies = len(policy_names)
    if num_policies == 0:
        raise ValueError('policy_names には少なくとも 1 つの方策名が必要です。')

    fig, axes = plt.subplots(num_policies, 1, figsize=(14, 3.6 * num_policies), sharex=True)
    if num_policies == 1:
        axes = [axes]

    for ax, policy_name in zip(axes, policy_names):
        shares = results[policy_name]['z_state_share_by_step']
        num_steps = results[policy_name]['num_steps']
        steps = list(range(1, num_steps + 1))
        ax.stackplot(
            steps,
            shares['z_B'],
            shares['z_H'],
            shares['z_L'],
            labels=['z_B (bench)', 'z_H (high)', 'z_L (low)'],
            alpha=0.9,
        )
        ax.set_ylim(0.0, 1.0)
        ax.set_ylabel('Share')
        ax.set_title(f'State composition by step: {policy_name}')
        ax.legend(loc='upper right')

    axes[-1].set_xlabel('Step within episode')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    return fig, axes


print('Definitions loaded.')

In [ ]:
mdp = EngineerAssignmentMDP()
mdp.describe_problem()


In [ ]:
violations = mdp.validate_transition_probabilities()
print('transition violations:', violations)
print('num_states =', mdp.num_states)
print('states =', mdp.states)
print('actions =', mdp.actions)


## 価値反復・方策反復

目的関数は `.tex` と同様に**割引累積報酬**です。  
そのうえで後ろの評価セルでは、**100 ステップを 1 エピソードとするモンテカルロ評価**を複数エピソード回し、平均・標準偏差・標準誤差を見ます。

In [ ]:
vi = value_iteration(mdp)
pi = policy_iteration(mdp)
print('VI iterations:', vi['iterations'])
print('PI iterations:', pi['iterations'])
print('\nVI policy:')
print(policy_to_action_names(mdp, vi['policy']))
print('\nPI policy:')
print(policy_to_action_names(mdp, pi['policy']))


## ベースライン比較とモンテカルロ評価

`.tex` で記号だけ与えられていた閾値には、いったん以下を使います。
- `k_safe = 1`
- `k_H = 2`
- `k_L = 1`

さらに、評価は以下の条件で行います。
- 初期状態: `('z_B', 0)`
- 1エピソード: `100` ステップ
- エピソード数: `1000`

必要なら次のセルで変更してください。

In [ ]:
k_safe = 1
k_H = 2
k_L = 1

num_episodes = 1000
num_steps = 100
seed = 42
initial_state = ('z_B', 0)

results = compare_policies(
    mdp,
    num_episodes=num_episodes,
    num_steps=num_steps,
    initial_state=initial_state,
    seed=seed,
    k_safe=k_safe,
    k_H=k_H,
    k_L=k_L,
)

table = results_to_table(results)
for row in table:
    print(row)

In [ ]:
fig, axes = plot_policy_comparison(results)
fig

## ステップごとの状態構成比（積み上げグラフ）

各ステップで、エピソード群のうちどれだけが `z_B` / `z_H` / `z_L` にいるかを積み上げ表示します。  
これにより、**待機から稼働へ移る速さ**や、**高単価案件にどれだけ乗れているか**の時間推移が見えます。

In [ ]:
fig, axes = plot_state_composition_by_step(results)
fig